# 05 — Reinforcement Learning: PPO Training
**Goal**: Train the SFT model with PPO and sandbox execution rewards. Two paper arms:

| Arm | `reward_mode` | Checkpoint |
|---|---|---|
| RQ3 dense | `dense` (partial credit) | `./checkpoints/ppo_dense/final` |
| RQ4 binary | `binary` (AC→1 else 0) | `./checkpoints/ppo_binary/final` |

Shared HPs: frozen SFT ref (`init_kl_coef=0.05`, `target_kl=6.0`, `kl_penalty=abs`, LR `1e-6`), fixed `seed`, packed multi-test partial credit for dense mode.

**Paper runs**: set `MAX_STEPS=100` (or higher) below. `MAX_STEPS=10` is diagnostic only.

---


## Step 1: Environment Setup & Checkpoint Resolution

In [ ]:
!pip install -q "trl<0.12.0" peft transformers datasets bitsandbytes accelerate

import sys, os, shutil

# 1. Pull latest code from junior-A branch
repo_root = os.path.abspath(os.getcwd())
if os.path.isdir('/kaggle/working'):
    !rm -rf /kaggle/working/src
    !git clone -b junior-A https://github.com/Oin19/self-correction-llm-rl.git temp_repo
    !cp -r temp_repo/src /kaggle/working/src
    GIT_SHA = !git -C temp_repo rev-parse HEAD
    GIT_SHA = GIT_SHA[0] if GIT_SHA else ''
    print(f'Cloned junior-A @ {GIT_SHA}')
    !rm -rf temp_repo
    repo_root = '/kaggle/working'

# 2. Path setup (expander/KL fixes now live in repo src/training/ppo.py)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path initialized: {repo_root}")

# 3. Uninstall torchao to prevent PEFT version conflicts
!pip uninstall -y torchao

# 4. Resolve & Copy SFT Checkpoint
SFT_PATH = './checkpoints/sft/final'
if not os.path.exists(SFT_PATH) and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'adapter_config.json' in files and 'sft' in root.lower():
            dest = '/kaggle/working/checkpoints/sft/final'
            os.makedirs(dest, exist_ok=True)
            for item in os.listdir(root):
                s = os.path.join(root, item)
                d = os.path.join(dest, item)
                if os.path.isfile(s): shutil.copy2(s, d)
                elif os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
            SFT_PATH = dest
            print(f"SUCCESS: SFT checkpoint copied to '{dest}'!")
            break

print(f"SFT Checkpoint ready at: {SFT_PATH}")

if 'GIT_SHA' not in dir():
    GIT_SHA = ''


## Step 2: Load APPS Dataset & Tokenizer

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from src.training.ppo import run_ppo_training, normalize_tests

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading APPS training dataset for PPO (1,000 problem pairs)...")
apps = load_dataset('codeparrot/apps', revision='refs/convert/parquet', split='train[:1000]')
# load_from_cache_file=False: stale HF cache can yield empty apps_clean after normalize_tests changes
apps_clean = apps.filter(lambda x: len(normalize_tests(x)) > 0, load_from_cache_file=False)
if len(apps_clean) == 0:
    sample = apps[0]
    raise ValueError(f'Cell4 filter kept 0/{len(apps)} rows; columns={sorted(sample.keys())}')
print(f"Prepared {len(apps_clean)} APPS problems with valid benchmark tests for PPO rollout.")

## Step 3: Pre-Flight Reward Unit Test (dense + binary)

In [ ]:
from src.execution.executor import PythonSandbox
from src.rewards.execution_reward import score_rollout_reward

sandbox = PythonSandbox(default_timeout=5.0, max_memory_mb=1024.0)
tests = [{"assertion": "assert add(1,2) == 3"}] * 5 + [{"assertion": "assert add(2,3) == 5"}] * 5
cases = {
    "perfect": "def add(a,b): return a+b",
    "partial": "def add(a,b): return a+b if a==1 else 0",
    "bad": "def add(a,b): return a+",
}
expected_dense = {"perfect": 1.0, "partial": 0.5, "bad": -0.2}
expected_binary = {"perfect": 1.0, "partial": 0.0, "bad": 0.0}
for name, code in cases.items():
    result = sandbox.run_tests(code, tests)
    d = score_rollout_reward(result, reward_mode="dense")
    b = score_rollout_reward(result, reward_mode="binary")
    print(f"{name}: status={result.status} passed={result.passed_tests}/{result.total_tests} dense={d:.3f} binary={b:.3f}")
    assert abs(d - expected_dense[name]) < 1e-9, name
    assert abs(b - expected_binary[name]) < 1e-9, name
print("PASS: dense partial + binary AC-only preflight verified.")


## Step 4: Run PPO Training (dense for RQ3 / binary for RQ4)

Flip `REWARD_MODE` and re-run for the second arm. Paper default `MAX_STEPS=100`.

In [ ]:
# Paper arms: REWARD_MODE = "dense" -> ./checkpoints/ppo_dense
#              REWARD_MODE = "binary" -> ./checkpoints/ppo_binary
REWARD_MODE = "dense"   # or "binary"
MAX_STEPS = 100         # use 10 only for smoke/diagnostic
SEED = 42
OUTPUT_DIR = f"./checkpoints/ppo_{REWARD_MODE}"

print(f"=== PPO Training | mode={REWARD_MODE} steps={MAX_STEPS} seed={SEED} -> {OUTPUT_DIR} ===")

ppo_trainer = run_ppo_training(
    sft_model_path=SFT_PATH,
    tokenizer=tokenizer,
    dataset=apps_clean,
    output_dir=OUTPUT_DIR,
    num_epochs=1,
    learning_rate=1e-6,
    batch_size=2,
    mini_batch_size=1,
    gradient_accumulation_steps=2,
    init_kl_coef=0.05,
    target_kl=6.0,
    max_steps=MAX_STEPS,
    reward_mode=REWARD_MODE,
    seed=SEED,
    commit_sha=GIT_SHA if 'GIT_SHA' in dir() else '',
)

print(f"\nPPO Training completed! Saved to {OUTPUT_DIR}/final")


## Step 5: Checkpoint Verification & Reload Test

In [ ]:
import os
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

PPO_PATH = f"./checkpoints/ppo_{REWARD_MODE}/final"
print(f"=== Checkpoint Reload Verification: {PPO_PATH} ===")
if os.path.isfile(os.path.join(PPO_PATH, "adapter_config.json")):
    import json as _json
    meta_path = os.path.join(PPO_PATH, "ppo_metadata.json")
    if os.path.isfile(meta_path):
        with open(meta_path, encoding="utf-8") as f:
            print("metadata:", _json.load(f))
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True
    )
    ppo_model = PeftModel.from_pretrained(base_model, PPO_PATH)
    print("SUCCESS: Reloaded PPO model successfully into PeftModel!")
else:
    raise FileNotFoundError(f"PPO checkpoint not found at {PPO_PATH}")
